# Piloto de audiolivro adaptado: Reinações de Narizinho

Narra, numa GPU grátis do Google Colab, os capítulos 1 a 12 de *Reinações de Narizinho*, numa adaptação para crianças e adolescentes com deficiência visual, síndrome de Down, autismo e TDAH. O guia de adaptação e de narração está no `GUIA.md` da pasta Audiolivros.

**Antes de começar:** menu `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4`. Depois rode as células de cima para baixo. Rodar de novo qualquer célula é seguro.

**Saída** (no seu Google Drive, em `MyDrive/Audiolivros/narizinho/capNN/`), para cada capítulo:
- `capNN_A_so_voz.mp3`: só a narração, para quem é sensível a sons;
- `capNN_B_com_ambiente.mp3`: narração com vinheta suave, alguns sons curtos e baixinhos nas pausas (sonoplastia leve) e, nas cenas do riacho, som de água bem baixinho.

Escolha os capítulos na célula 2 (`CAPITULOS_PARA_GERAR`). As vozes ficam em `narizinho/vozes/` e são as mesmas em todos os capítulos.

**Licença:** a voz padrão (OmniVoice) é de uso **não comercial** (CC-BY-NC). Serve para um piloto gratuito. As vozes são sintéticas e originais, sem clonar ninguém.

**Sotaque:** o modelo fala português, mas às vezes puxa para o sotaque de Portugal. Se acontecer, troque a `semente` da voz na célula 5. Outra saída é usar uma gravação de 5 a 10 segundos de alguém que **autorizou por escrito** o uso da própria voz (campo `amostra`).

In [ ]:
# ── 1. Conferir a GPU ───────────────────────────────────────────────────────
import shutil, subprocess
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("ATENÇÃO: sem GPU. Vá em Ambiente de execução -> Alterar o tipo de ambiente "
          "de execução -> GPU T4, e rode o notebook de novo do início.")

In [ ]:
# ── 2. Google Drive e pastas ────────────────────────────────────────────────
import os
from google.colab import drive
drive.mount("/content/drive")

CAPITULOS_PARA_GERAR = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]   # quais capítulos narrar (ex.: [2, 3])

LIVRO = "/content/drive/MyDrive/Audiolivros/narizinho"
PASTA_VOZES = os.path.join(LIVRO, "vozes")   # uma fala de referência por personagem
os.makedirs(PASTA_VOZES, exist_ok=True)
print("Os arquivos vão para:", LIVRO)

In [ ]:
# ── 3. Instalar o VoiceStudio (primeira vez: ~5-8 min) ──────────────────────
import os, shutil, subprocess, sys

REPO_URL = "https://github.com/sdpaula36-stack/VoiceStudio"
REPO_DIR = "/content/VoiceStudio"
os.environ.setdefault("UV_HTTP_TIMEOUT", "300")

def run(cmd, *, cwd=None, what=""):
    shown = cmd if isinstance(cmd, str) else " ".join(cmd)
    print(f"\n$ {shown}")
    if subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str)).returncode != 0:
        raise SystemExit(f"\nFALHOU: {what or shown}. Veja o erro acima; rodar de novo é seguro.")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    run(["git", "-C", REPO_DIR, "pull", "--ff-only"], what="git pull")
else:
    run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], what="git clone")

run("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1", what="apt-get install")
if not shutil.which("uv"):
    run([sys.executable, "-m", "pip", "install", "-q", "uv"], what="pip install uv")
run(["uv", "pip", "install", "--system", "--no-cache",
     "--constraint", "deploy/torch-constraints.txt",
     "--override", "deploy/uv-overrides.txt", "."],
    cwd=REPO_DIR, what="instalação do backend")

from huggingface_hub import snapshot_download
print("Baixando o modelo de voz (alguns GB, só na primeira vez)...")
snapshot_download("k2-fsa/OmniVoice")
print("Pronto.")

In [ ]:
# ── 4. Ligar o servidor do VoiceStudio ──────────────────────────────────────
import json, os, subprocess, sys, time, urllib.request

PORT = 3900
BASE = f"http://127.0.0.1:{PORT}"
LOG_PATH = "/content/voicestudio_backend.log"

def health():
    try:
        with urllib.request.urlopen(f"{BASE}/health", timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

info = health()
if not info:
    subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
    env = os.environ.copy()
    env.update(OMNIVOICE_SERVER_MODE="1", PYTHONUNBUFFERED="1")
    env.setdefault("OMNIVOICE_DATA_DIR", "/content/omnivoice_data")
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "main:app", "--app-dir", "backend",
         "--host", "127.0.0.1", "--port", str(PORT)],
        cwd="/content/VoiceStudio", env=env,
        stdout=open(LOG_PATH, "ab"), stderr=subprocess.STDOUT)
    deadline = time.time() + 300
    while time.time() < deadline and proc.poll() is None and not info:
        print(".", end="", flush=True)
        time.sleep(3)
        info = health()
    print()
    if not info:
        raise SystemExit("O servidor não ligou em 5 min. Final do log:\n"
                         + "".join(open(LOG_PATH, errors="replace").readlines()[-40:]))
print("Servidor ligado:", info)

In [ ]:
# ── 4.5. Aceitar as licenças dos modelos (obrigatório antes de gerar) ───────
# Sem este passo o /generate falha. Rodar esta célula aceita as licenças em seu nome:
# OmniVoice é CC-BY-NC (uso NÃO comercial: o áudio não pode ser vendido).
import requests
BASE = "http://127.0.0.1:3900"

for repo in ("k2-fsa/OmniVoice", "eustlb/higgs-audio-v2-tokenizer"):
    st = requests.get(f"{BASE}/models/licenses/acceptance/{repo}", timeout=30).json()
    if st.get("accepted"):
        print(f"[ok] {repo} já aceito")
        continue
    r = requests.post(
        f"{BASE}/models/licenses/accept",
        json={"repo_id": repo, "fingerprint": st["fingerprint"], "accepted": True},
        timeout=30,
    )
    r.raise_for_status()
    print(f"[aceito] {repo} ({st.get('license')})")

## Roteiro

Cada trecho é `[personagem, texto, velocidade, pausa depois em segundos]` e, às vezes, um quinto item: o **efeito** que toca na pausa, só na versão B (`transicao`, `magia`, `agua`, `galope`, `porta`, `zumbido`, `festa`). Cada capítulo tem `"ambiente"`: `"riacho"` (água baixinha durante a fala) ou `"nenhum"`. Regras em `SONOPLASTIA.md`.

- **velocidade:** 1.0 é normal; até 1.1 na ação; 0.85 a 0.9 no suspense e no sono.
- **pausa:** o silêncio depois do trecho. Pausas maiores antes das revelações prendem a atenção.
- **voz** aceita só estes termos: `male`/`female`; `child`, `teenager`, `young adult`, `middle-aged`, `elderly`; `very low pitch` … `very high pitch`; `whisper`.
- A voz de cada personagem nasce da fala mais longa dele no **primeiro** capítulo em que aparece, e é a mesma em todos os outros.

Edite à vontade. Os trechos que você mudar são gerados de novo na célula 7.

In [ ]:
# ── 5. Roteiro (EDITE AQUI) ─────────────────────────────────────────────────
# "amostra" (opcional, em cada voz): caminho de um .wav de 5-10 s de alguém que
# AUTORIZOU por escrito o uso da voz. Nunca use a voz de um dublador ou artista.
VOZES = {'narrador': {'voz': 'male, young adult, moderate pitch', 'semente': 201},
 'narizinho': {'voz': 'female, child, high pitch', 'semente': 211},
 'principe': {'voz': 'male, teenager, high pitch', 'semente': 221},
 'major': {'voz': 'male, middle-aged, very low pitch', 'semente': 231},
 'aranha': {'voz': 'female, middle-aged, high pitch', 'semente': 241},
 'caramujo': {'voz': 'male, elderly, low pitch', 'semente': 251},
 'emilia': {'voz': 'female, young adult, very high pitch', 'semente': 261},
 'nastacia': {'voz': 'female, middle-aged, low pitch', 'semente': 271},
 'dona_benta': {'voz': 'female, elderly, moderate pitch', 'semente': 281},
 'rabico': {'voz': 'male, teenager, low pitch', 'semente': 301},
 'formiga': {'voz': 'female, young adult, high pitch', 'semente': 311},
 'pedrinho': {'voz': 'male, child, moderate pitch', 'semente': 291},
 'tom_mix': {'voz': 'male, young adult, low pitch', 'semente': 321},
 'libelula': {'voz': 'female, child, very high pitch', 'semente': 361},
 'abelha': {'voz': 'female, teenager, high pitch', 'semente': 351},
 'rainha': {'voz': 'female, middle-aged, moderate pitch', 'semente': 341},
 'visconde': {'voz': 'male, elderly, moderate pitch', 'semente': 371}}

CAPITULOS = {
    1: {"titulo": "Um peixinho no nariz", "ambiente": "riacho", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo um: Um peixinho no nariz.", 0.95, 1.2],
        ["narrador", "Numa casinha branca, no meio do mato, fica o Sítio do Picapau Amarelo.", 1.0, 0.5],
        ["narrador", "Lá mora uma senhora de óculos, muito boazinha, chamada Dona Benta.", 1.0, 0.5],
        ["narrador", "Dona Benta é avó de uma menina de sete anos. O nome dela é Lúcia.", 1.0, 0.6],
        ["narrador", "Mas ninguém chama a menina de Lúcia. Sabe por quê?", 1.0, 0.8],
        ["narrador", "Porque o nariz dela é pequenininho e virado para cima. Um nariz arrebitado!", 1.05, 0.5],
        ["narrador", "Por isso, todo mundo chama a menina de... Narizinho!", 1.0, 0.9],
        ["narrador", "No sítio também mora Tia Nastácia. Ela faz os melhores bolinhos do mundo e cuida de todos com muito carinho.", 1.0, 0.5],
        ["narrador", "Foi Tia Nastácia quem fez a melhor amiga de Narizinho: uma boneca de pano chamada Emília.", 1.0, 0.5],
        ["narrador", "Emília tem olhos de linha preta e sobrancelhas tão lá no alto que parece estar sempre espantada.", 1.0, 0.6],
        ["narrador", "Só tem um problema. Emília não fala. Nem uma palavrinha.", 0.95, 0.8],
        ["narrador", "Pelo menos... por enquanto.", 0.9, 1.4],
        ["narrador", "Atrás da casa passa um riacho de água bem clarinha. Narizinho adora ir lá.", 1.0, 0.5],
        ["narrador", "Toda tarde ela pega Emília no colo, senta na beira da água e joga migalhas de pão para os peixinhos.", 1.0, 0.5],
        ["narrador", "Os peixinhos já conhecem a menina. Quando ela chega, eles vêm nadando, todos juntos: plic, plic, plic!", 1.05, 0.8],
        ["narrador", "Naquela tarde, o sol estava quentinho. O vento balançava as folhas, bem devagar...", 0.9, 0.6],
        ["narrador", "E Narizinho foi ficando com sono... com sono...", 0.85, 0.8],
        ["narrador", "Até que fechou os olhos e dormiu.", 0.85, 1.8],
        ["narrador", "De repente, alguma coisa fez cócegas no nariz dela!", 1.1, 0.5],
        ["narrador", "Era um peixinho! Um peixinho pequeno, com roupa de escamas brilhantes e uma coroa dourada na cabeça.", 1.05, 0.5],
        ["narrador", "E sabe o que ele estava fazendo? Estava em pé, bem em cima do nariz de Narizinho!", 1.05, 0.7],
        ["principe", "Que montanha mais esquisita! É macia, é quentinha... e tem dois buracos que sopram vento!", 1.0, 0.7],
        ["narrador", "Narizinho ficou quietinha, quietinha, segurando a risada.", 0.95, 0.6],
        ["narrador", "Então, do meio da água, saiu um caranguejo enorme, com as garras abertas. Era o Major Agarra-e-Não-Larga-Mais, o guarda do príncipe.", 1.0, 0.5],
        ["major", "Alteza! Cuidado! Essa montanha pode ser perigosa!", 1.0, 0.5],
        ["principe", "Bobagem, Major. Montanha não faz mal a ninguém.", 1.0, 0.7],
        ["narrador", "Nessa hora, Narizinho não aguentou mais... e espirrou! Atchim!", 1.1, 0.4],
        ["narrador", "O peixinho saiu voando e caiu de costas na grama!", 1.1, 0.4],
        ["principe", "Socorro! A montanha está viva!", 1.1, 0.6],
        ["narizinho", "Calma! Eu não sou montanha. Eu sou uma menina! Meu nome é Narizinho.", 1.0, 0.7],
        ["narrador", "O peixinho se levantou, arrumou a coroa e fez uma reverência bem educada.", 1.0, 0.5],
        ["principe", "Muito prazer, menina. Eu sou o Príncipe Escamado, rei do Reino das Águas Claras.", 0.95, 0.4],
        ["principe", "Desculpe ter subido no seu nariz. Ele é tão bonitinho que eu pensei que fosse uma ilha!", 1.0, 0.7],
        ["narrador", "Narizinho riu. Ninguém nunca tinha elogiado o nariz dela daquele jeito.", 1.0, 0.6],
        ["narizinho", "E onde fica esse tal Reino das Águas Claras?", 1.0, 0.5],
        ["principe", "Lá no fundo do riacho. Tem palácio de conchas, peixes de todas as cores e muitas coisas que você nunca viu.", 1.0, 0.4],
        ["principe", "Quer conhecer?", 0.95, 1.0],
        ["narrador", "Narizinho olhou para a água. Depois olhou para Emília. Emília, como sempre, não disse nada.", 0.95, 0.6],
        ["narizinho", "Eu quero! Mas a Emília vai junto.", 1.05, 0.5],
        ["principe", "Então segure a minha nadadeira. E não solte por nada!", 1.05, 0.6],
        ["narrador", "Narizinho segurou a nadadeira do príncipe, apertou Emília contra o peito...", 1.0, 0.4],
        ["narrador", "E, tchibum! Mergulhou no riacho!", 1.1, 1.6],
        ["narrador", "Lá embaixo, tudo era azul e brilhante. E no palácio de conchas, Narizinho ia conhecer um médico muito esquisito: o Doutor Caramujo.", 0.9, 0.6],
        ["narrador", "O Doutor Caramujo guardava umas pílulas mágicas. E uma dessas pílulas ia mudar a vida da Emília para sempre...", 0.85, 1.0],
        ["narrador", "Mas isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    2: {"titulo": "O palácio de conchas", "ambiente": "riacho", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo dois: O palácio de conchas.", 0.95, 1.2],
        ["narrador", "No capítulo passado, Narizinho dormiu na beira do riacho e acordou com um peixinho de coroa em cima do nariz.", 1.0, 0.4],
        ["narrador", "Era o Príncipe Escamado. Ele convidou Narizinho para conhecer o Reino das Águas Claras. E ela mergulhou, com Emília no colo!", 1.0, 1.2],
        ["narrador", "Debaixo da água, Narizinho percebeu uma coisa incrível. Ela conseguia respirar! E nem estava molhada!", 1.05, 0.6],
        ["narrador", "Em volta dela passavam peixes amarelos, peixes azuis, peixes listrados. Uns grandes, outros do tamanho de um grão de feijão.", 1.0, 0.5],
        ["narrador", "E lá na frente, brilhando como um castelo de festa, estava o palácio de conchas.", 0.95, 0.8],
        ["narizinho", "Que lindo! Parece feito de pérolas!", 1.05, 0.6],
        ["principe", "Seja bem-vinda ao meu palácio, Narizinho. Aqui você é a minha convidada de honra.", 1.0, 0.7],
        ["narrador", "Na porta do palácio, quem estava de guarda? Ele mesmo: o Major Agarra-e-Não-Larga-Mais, com as garras para cima.", 1.0, 0.4],
        ["major", "Alto lá! Quem vem com o príncipe pode passar. Mas a boneca de pano fica de olho fechado!", 1.0, 0.6],
        ["narrador", "Narizinho riu. Emília não podia fechar os olhos, porque eles eram costurados com linha preta.", 1.0, 0.9],
        ["narrador", "Lá dentro, o príncipe levou Narizinho até uma salinha cheia de fios e agulhas.", 1.0, 0.4],
        ["narrador", "Era o ateliê da Dona Aranha, a costureira mais famosa do reino. Ela tinha oito patas, e cada pata segurava uma agulha diferente.", 1.0, 0.5],
        ["aranha", "Ora, ora, uma menina de verdade! Fique paradinha, que eu vou fazer um vestido de festa para você.", 1.0, 0.6],
        ["narrador", "E as oito patas começaram a costurar, depressa, depressa, depressa! Zip, zip, zip!", 1.1, 0.5],
        ["narrador", "Em um minuto, o vestido estava pronto. Era azul como o céu, com bolinhas de espuma do mar.", 1.0, 0.5],
        ["narizinho", "Dona Aranha, é o vestido mais bonito que eu já vi!", 1.05, 0.5],
        ["aranha", "Fiz com fio de teia e luz da lua. Cuidado para não sujar de lama!", 1.0, 1.0],
        ["narrador", "Depois, o príncipe disse que queria apresentar um amigo muito importante.", 1.0, 0.4],
        ["narrador", "Bateram numa porta pequenininha. Toc, toc, toc.", 0.95, 0.8],
        ["narrador", "A porta abriu bem devagar... e apareceu um caramujo de óculos, com uma maleta preta e cara de sabido.", 0.9, 0.5],
        ["caramujo", "Boa tarde, boa tarde. Eu sou o Doutor Caramujo, o médico do reino. Quem está doente aqui?", 0.95, 0.6],
        ["principe", "Ninguém está doente, doutor. Mas a boneca da Narizinho não fala.", 1.0, 0.5],
        ["caramujo", "Não fala? Hum... Deixe-me ver.", 0.95, 0.6],
        ["narrador", "O doutor olhou a boca de Emília com uma lupa. Olhou de um lado. Olhou do outro.", 0.95, 0.5],
        ["caramujo", "Já sei o que é. Esta boneca tem falta de fala. Mas eu tenho o remédio certo!", 1.0, 0.6],
        ["narrador", "Ele abriu a maleta preta e tirou um vidrinho cheio de bolinhas coloridas.", 1.0, 0.4],
        ["caramujo", "São as minhas pílulas falantes. Uma só, e qualquer um fala pelos cotovelos!", 1.0, 0.8],
        ["narrador", "Narizinho segurou Emília bem firme. O doutor pegou uma pílula e colocou na boca da boneca.", 0.95, 0.5],
        ["narrador", "Emília engoliu. Gulp!", 0.95, 1.2],
        ["narrador", "Todo mundo ficou olhando. O príncipe, a Dona Aranha, o Major, o doutor... e Narizinho, de olhos arregalados.", 0.9, 0.8],
        ["narrador", "Um segundo. Dois segundos. Três segundos...", 0.85, 1.0],
        ["narrador", "E então a boca de pano da Emília começou a mexer.", 0.85, 1.2],
        ["narrador", "Qual será a primeira palavra da Emília? Você vai descobrir no próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    3: {"titulo": "Emília fala!", "ambiente": "riacho", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo três: Emília fala!", 0.95, 1.2],
        ["narrador", "No capítulo passado, Narizinho ganhou um vestido da Dona Aranha e conheceu o Doutor Caramujo.", 1.0, 0.4],
        ["narrador", "O doutor deu para a Emília uma pílula falante. E a boca de pano da boneca começou a mexer...", 0.95, 1.2],
        ["emilia", "Estou com um gosto horrível de sapo na boca!", 1.05, 0.9],
        ["narrador", "Todo mundo levou um susto. A Emília estava falando!", 1.05, 0.5],
        ["narizinho", "Emília! Você fala! Você fala de verdade!", 1.1, 0.5],
        ["emilia", "Claro que falo. Eu sempre pensei um monte de coisas. Só faltava a boca funcionar!", 1.05, 0.6],
        ["narrador", "E a Emília não parou mais. Falava, falava, falava, igual a uma torneira aberta.", 1.05, 0.5],
        ["emilia", "Esse príncipe é muito molhado. Esse caranguejo é muito bravo. E esse doutor tem cara de quem come alface no café da manhã!", 1.1, 0.6],
        ["caramujo", "Hum. Acho que a pílula era forte demais.", 0.95, 0.6],
        ["narrador", "Narizinho ficou um pouco envergonhada. Mas também não conseguia parar de rir.", 1.0, 0.5],
        ["narizinho", "Emília, fale com educação! Eles são nossos amigos.", 1.0, 0.5],
        ["emilia", "Está bem, está bem. Muito obrigada pela pílula, senhor doutor. Mas da próxima vez, faça com gosto de goiabada!", 1.05, 0.9],
        ["narrador", "O príncipe achou tudo muito engraçado e resolveu fazer uma festa.", 1.0, 0.4],
        ["principe", "Hoje tem baile no palácio! Para comemorar a boneca que aprendeu a falar!", 1.05, 0.6],
        ["narrador", "Os peixes tocaram música com as bolhas: blub, blub, blub. Os camarões dançaram em fila. E até o Major mexeu as garras no ritmo.", 1.05, 0.5],
        ["narrador", "Narizinho dançou com o príncipe, de vestido azul. E Emília dançou sozinha, girando, girando, e falando sem parar.", 1.0, 0.9],
        ["narrador", "No meio da festa, de repente, Narizinho ouviu uma voz que vinha lá de cima. De muito longe.", 0.9, 0.8],
        ["nastacia", "Narizinho! Vem lanchar, menina! Tem bolinho quentinho!", 0.95, 0.8],
        ["narizinho", "É a Tia Nastácia! Preciso voltar para casa.", 1.0, 0.5],
        ["principe", "Já vai? Mas a festa mal começou!", 1.0, 0.5],
        ["narizinho", "Eu volto, prometo. Obrigada por tudo, príncipe!", 1.0, 0.8],
        ["narrador", "Narizinho fechou os olhos. Quando abriu de novo... estava deitada na grama, na beira do riacho.", 0.9, 0.8],
        ["narrador", "Será que tinha sido tudo um sonho?", 0.85, 1.0],
        ["narrador", "Ela olhou para a boneca no colo. E a boneca olhou de volta para ela.", 0.9, 0.8],
        ["emilia", "Que foi? Nunca viu boneca falar, não?", 1.05, 1.0],
        ["narrador", "Não tinha sido sonho! Narizinho saiu correndo para casa, com a Emília falando sem parar.", 1.1, 0.5],
        ["narrador", "E o que será que a Dona Benta e a Tia Nastácia vão dizer quando ouvirem a Emília falar?", 0.95, 0.6],
        ["narrador", "Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    4: {"titulo": "A boneca que fala", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo quatro: A boneca que fala.", 0.95, 1.2],
        ["narrador", "No capítulo passado, a Emília tomou a pílula falante do Doutor Caramujo e começou a falar sem parar.", 1.0, 0.4],
        ["narrador", "Depois, Narizinho voltou do Reino das Águas Claras para o sítio, com a Emília no colo.", 1.0, 1.0],
        ["narrador", "Narizinho entrou correndo em casa. Na sala estava Dona Benta, a avó dela, sentada na cadeirinha de balanço.", 1.0, 0.5],
        ["dona_benta", "Narizinho, tenho uma novidade! Seu primo Pedrinho vem passar uns tempos aqui no sítio.", 1.0, 0.5],
        ["narizinho", "O Pedrinho! Que bom, que bom, que bom!", 1.1, 0.5],
        ["narrador", "Narizinho deu três pulos de alegria. Pedrinho é um menino de dez anos, que mora na cidade com a mãe.", 1.0, 0.6],
        ["dona_benta", "Mas antes, arrume essa boneca. Ela está toda amassada e com o vestido molhado!", 1.0, 0.6],
        ["emilia", "A culpa não é minha, Dona Benta! Foi a Narizinho que me levou para o fundo do riacho.", 1.05, 1.0, "magia"],
        ["narrador", "Dona Benta levou um susto tão grande que quase caiu da cadeira. A boneca de pano estava falando!", 1.05, 0.5],
        ["dona_benta", "Tia Nastácia! Corra aqui! Venha ver uma coisa que nunca se viu!", 1.05, 0.5],
        ["narrador", "Tia Nastácia veio da cozinha, enxugando as mãos no avental.", 1.0, 0.4],
        ["nastacia", "Boneca falando, Dona Benta? Isso não existe. A Narizinho está brincando com a senhora.", 0.95, 0.6],
        ["emilia", "Não estou brincando coisa nenhuma! Eu falo, sim. O doutor me deu uma pílula, e agora vou falar a vida inteira!", 1.05, 0.7],
        ["narrador", "Tia Nastácia ficou de boca aberta. Ela mesma tinha costurado aquela boneca, com um paninho bem simples.", 1.0, 0.5],
        ["nastacia", "Credo! Se eu soubesse que ela ia falar, tinha feito a Emília de seda!", 1.0, 0.9],
        ["narrador", "No dia seguinte chegou uma carta da mãe de Pedrinho. Ele ia chegar dali a uma semana.", 1.0, 0.5],
        ["narizinho", "Uma semana inteira? Que demora!", 1.0, 0.5],
        ["narrador", "Enquanto isso, a Emília só queria ouvir histórias. Pedia a história de tudo: do tapete, do relógio, do armário.", 1.0, 0.5],
        ["emilia", "Dona Benta, conte a história do Pedrinho!", 1.05, 0.5],
        ["dona_benta", "Pedrinho não tem história, Emília. Ele nunca saiu de casa.", 1.0, 0.5],
        ["emilia", "Ora essa! O livro de capa vermelha também nunca saiu de casa. E tem mais de dez histórias dentro!", 1.05, 0.8],
        ["narrador", "Dona Benta riu. A Emília falava muita bobagem, mas sempre tinha uma resposta.", 1.0, 0.8],
        ["narrador", "Até agora: a Emília fala, Dona Benta e Tia Nastácia levaram um susto, e Pedrinho vai chegar daqui a uma semana.", 0.95, 1.0, "transicao"],
        ["narrador", "Para a semana passar mais depressa, Narizinho tinha uma coisa boa: era tempo de jabuticaba!", 1.05, 0.5],
        ["narrador", "Jabuticaba é uma fruta redonda e preta, do tamanho de uma bolinha de gude. Ela cresce grudada no tronco da árvore.", 1.0, 0.5],
        ["narrador", "A casca é lisinha. Dentro tem uma polpa branca, muito doce.", 1.0, 0.6],
        ["narrador", "Narizinho subia na jabuticabeira e escolhia as mais bonitas. Mordia, e fazia tloc! Engolia o caldo e cuspia o caroço: pluf!", 1.05, 0.5],
        ["narrador", "Lá embaixo esperava o Rabicó, um leitãozinho comilão, de rabinho bem curtinho. Ele pegava as cascas no chão: nhoc!", 1.05, 0.5],
        ["narrador", "E a música da jabuticabeira era assim: tloc, pluf, nhoc! Tloc, pluf, nhoc!", 1.05, 0.8],
        ["narrador", "Mas a árvore tinha outros fregueses: as vespas. As vespas furavam as frutas com o ferrão e entravam dentro, para chupar o doce.", 1.0, 0.6],
        ["narrador", "Uma tarde, Narizinho pôs na boca uma jabuticaba furadinha. Ela não viu que tinha uma vespa lá dentro.", 0.95, 0.8],
        ["narrador", "Por que será que, dessa vez, não fez tloc?", 0.95, 2.5],
        ["narrador", "Porque a vespa picou a ponta da língua dela! Ai, ai, ai!", 1.05, 0.6],
        ["narrador", "Tia Nastácia ouviu o grito e veio correndo. Sentou Narizinho no colo.", 1.0, 0.4],
        ["nastacia", "Calma, menina. Põe a língua para fora, que eu tiro o ferrão. Dói, mas passa.", 0.95, 0.6],
        ["narrador", "Com uma pinça, Tia Nastácia tirou o ferrãozinho, bem devagar. Narizinho chorou um pouco, de língua inchada. Depois a dor passou.", 0.95, 0.6],
        ["narrador", "Ela foi contar tudo para a Emília.", 1.0, 0.4],
        ["emilia", "Bem feito! Se fosse eu, olhava cada fruta com o binóculo da Dona Benta antes de comer.", 1.05, 0.8],
        ["narrador", "Narizinho não aguentou e deu uma gargalhada. E daí a pouco já estava na árvore de novo: tloc, pluf, nhoc!", 1.05, 1.0],
        ["narrador", "Resumindo: Narizinho levou uma picada de vespa, Tia Nastácia cuidou dela, e a Emília deu um conselho bem maluco.", 0.95, 0.8],
        ["narrador", "Mas naquela noite, Narizinho esqueceu a Emília debaixo da jabuticabeira, sozinha no escuro.", 0.9, 0.8],
        ["narrador", "E lá, a boneca viu uma coisa muito curiosa acontecer com aquela vespa...", 0.9, 1.0],
        ["narrador", "O que será que a Emília viu? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    5: {"titulo": "A condessa pescadora", "ambiente": "riacho", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo cinco: A condessa pescadora.", 0.95, 1.2],
        ["narrador", "No capítulo passado, a Emília deu um susto em Dona Benta e Tia Nastácia, porque agora ela fala.", 1.0, 0.4],
        ["narrador", "Depois, uma vespa picou a língua de Narizinho. E de noite, a menina esqueceu a Emília debaixo da jabuticabeira.", 1.0, 1.0],
        ["narrador", "Na hora de dormir, Narizinho lembrou da boneca.", 1.0, 0.4],
        ["narizinho", "Coitada da Emília! Deve estar com medo das corujas. Tia Nastácia, vai buscar a Emília para mim?", 1.0, 0.6],
        ["narrador", "Tia Nastácia foi e trouxe a boneca. A Emília estava molhada de orvalho, que são gotinhas de água que aparecem de noite nas folhas.", 1.0, 0.5],
        ["narrador", "E estava muito brava, porque tinha sido esquecida.", 1.0, 0.5],
        ["narizinho", "Desculpa, Emília. Eu faço um vestido novo para você. Cor-de-rosa, com bolinhas.", 1.0, 0.5],
        ["emilia", "Quero de saia bem comprida. Sujei o joelho e não quero que apareça.", 1.05, 0.5],
        ["narizinho", "É só lavar o joelho, ora.", 1.0, 0.5],
        ["emilia", "Nada disso! Por dentro eu sou recheada de macela. Se eu me molhar, fico mofada!", 1.05, 0.6],
        ["narrador", "Macela é uma florzinha seca, macia e cheirosa. Tia Nastácia usava macela para encher a boneca, como se fosse algodão.", 1.0, 0.6],
        ["emilia", "E um dia eu vou ser condessa. Quero ser a Condessa de Três Estrelinhas!", 1.05, 0.6],
        ["narrador", "Condessa é uma moça importante, quase uma princesa. Narizinho achou graça e pintou três estrelinhas na testa da boneca.", 1.0, 0.6],
        ["narizinho", "Pronto! De hoje em diante, você é a Condessa de Três Estrelinhas.", 1.0, 0.8],
        ["narrador", "Então a Emília contou o que viu lá fora, no escuro.", 1.0, 0.4],
        ["emilia", "A vespa que picou você ficou presa dentro da jabuticaba. E as formigas vieram buscar.", 1.0, 0.5],
        ["emilia", "Cada formiga pegou uma perninha e puxou, puxou, até o buraquinho onde elas moram. Era o enterro da vespa!", 1.05, 0.5],
        ["emilia", "Um besourinho quis fazer um discurso bem comprido. Mas as formigas ficaram com fome e foram embora sem ouvir!", 1.05, 0.8],
        ["narrador", "Narizinho riu tanto que até esqueceu da língua inchada.", 1.0, 1.0, "transicao"],
        ["narrador", "Até agora: a Emília virou condessa e contou o enterro da vespa. Agora vem uma pescaria!", 0.95, 0.8],
        ["narrador", "Uns dias depois, as jabuticabas acabaram. Tia Nastácia foi lavar roupa no riacho, e Narizinho foi junto, com a Emília no bolso do avental.", 1.0, 0.5],
        ["narizinho", "Tia Nastácia, faz um anzol para a Emília? Ela quer tanto pescar!", 1.0, 0.5],
        ["nastacia", "Era só o que faltava! Mas está bem, eu faço.", 1.0, 0.5],
        ["narrador", "Tia Nastácia dobrou um alfinete, que virou um anzolzinho. Amarrou numa linha e numa varinha do comprimento de uma régua.", 1.0, 0.5],
        ["narrador", "Narizinho amarrou a vara no braço da Emília. Depois sentou a boneca na beira da água, com uma pedra no colo para ela não cair.", 1.0, 0.5],
        ["narizinho", "Agora, Emília, fica quietinha. Se um peixe morder, você puxa a linha!", 1.0, 0.6],
        ["narrador", "Narizinho foi ajudar Tia Nastácia com a roupa. Mas, de repente... tchibum!", 1.05, 1.0, "agua"],
        ["narrador", "A Emília tinha caído dentro do riacho, com pedra e tudo!", 1.1, 0.5],
        ["narizinho", "Socorro, Tia Nastácia! A Emília está se afogando!", 1.1, 0.5],
        ["narrador", "Por que será que a Emília caiu na água?", 0.95, 2.5],
        ["narrador", "Porque um peixe mordeu o anzol e puxou a boneca para o meio do rio!", 1.05, 0.5],
        ["narrador", "Tia Nastácia pegou uma vara comprida e, com muito jeito, puxou a Emília para a beira.", 1.0, 0.5],
        ["narrador", "E, preso no anzol da boneca, vinha um peixe! Era uma traíra, um peixe comprido, de dentinhos pontudos, sacudindo o rabo sem parar.", 1.05, 0.6],
        ["nastacia", "Credo! Até parece mágica!", 1.0, 0.6],
        ["narrador", "Narizinho correu para casa com o peixe na mão.", 1.05, 0.4],
        ["narizinho", "Vovó, adivinha quem pescou esta traíra!", 1.05, 0.5],
        ["dona_benta", "Ora, foi você, minha filha.", 1.0, 0.4],
        ["narizinho", "Errou! Foi a Emília!", 1.1, 0.5],
        ["narrador", "Tia Nastácia chegou com a trouxa de roupa e confirmou: foi a boneca mesmo.", 1.0, 0.5],
        ["dona_benta", "Quanto mais a gente vive, mais aprende. Tenho mais de sessenta anos e todo dia esta neta me ensina uma coisa nova!", 1.0, 0.6],
        ["narrador", "E Tia Nastácia fritou a traíra para o jantar.", 1.0, 1.0],
        ["narrador", "Resumindo: a Condessa de Três Estrelinhas caiu no rio, mas pescou um peixe de verdade.", 0.95, 0.8],
        ["narrador", "Só que, depois do jantar, Narizinho lembrou: a Emília ainda estava encharcada!", 1.0, 0.6],
        ["narrador", "E, enquanto secava no sol, a Emília descobriu que entendia a língua das formigas...", 0.9, 1.0],
        ["narrador", "O que será que as formigas falam? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    6: {"titulo": "O presente da rainha das formigas", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo seis: O presente da rainha das formigas.", 0.95, 1.2],
        ["narrador", "No capítulo passado, a Emília virou a Condessa de Três Estrelinhas.", 1.0, 0.4],
        ["narrador", "Depois, ela caiu no riacho e pescou uma traíra. Mas ficou toda encharcada!", 1.0, 1.0],
        ["narrador", "Narizinho tirou a roupa molhada da Emília e deixou a boneca no sol, para secar.", 1.0, 0.5],
        ["emilia", "Eu não fico aqui sozinha! O Rabicó pode me comer. Tia Nastácia diz que ele come tudo o que encontra.", 1.05, 0.5],
        ["narizinho", "Está bem, eu fico com você. Mas só porque você pescou a traíra.", 1.0, 0.8],
        ["narrador", "O sol estava quente. No chão, passavam formiguinhas ruivas, quer dizer, de cor avermelhada. Iam e vinham, sem parar.", 1.0, 0.5],
        ["narizinho", "Que pena a gente não entender o que as formigas conversam.", 1.0, 0.5],
        ["emilia", "Você não entende. Eu entendo tudo!", 1.05, 0.5],
        ["narrador", "Duas formigas se encontraram e encostaram as anteninhas, que são dois fiozinhos que elas têm na cabeça.", 1.0, 0.5],
        ["emilia", "Uma disse: Achou o grilinho? A outra disse: Não. Então a primeira mandou procurar perto da pedra.", 1.05, 0.6],
        ["narrador", "Logo chegou uma terceira formiga, bem apressada.", 1.0, 0.4],
        ["emilia", "Essa aí disse que acharam uma minhoca gorda perto da porteira. E que precisam de ajuda para carregar.", 1.05, 0.6],
        ["narizinho", "Emília, você está inventando! Vou lá ver.", 1.0, 0.5],
        ["narrador", "Narizinho correu até a porteira. E não é que era verdade? Muitas formiguinhas puxavam juntas uma minhoca grande para o formigueiro.", 1.05, 0.6],
        ["narrador", "Por que será que as formigas trabalham todas juntas?", 0.95, 2.5],
        ["narrador", "Porque, sozinha, uma formiga não consegue carregar uma coisa tão pesada. Juntas, elas conseguem.", 1.0, 0.8],
        ["narrador", "Até agora: a Emília diz que entende a língua das formigas. E Narizinho começou a acreditar.", 0.95, 0.8],
        ["narrador", "De noite, as duas deitaram juntas na cama, como faziam desde que a Emília aprendeu a falar.", 1.0, 0.5],
        ["narizinho", "Emília, como é que você entende as formigas?", 1.0, 0.4],
        ["emilia", "Porque eu sou de pano.", 1.0, 0.5],
        ["narizinho", "Meu vestido também é de pano, e não entende nada!", 1.0, 0.4],
        ["emilia", "Então é porque eu sou de macela.", 1.0, 0.5],
        ["narizinho", "Este travesseiro também é de macela!", 1.0, 0.4],
        ["emilia", "Então... então eu não sei.", 0.95, 0.8],
        ["narrador", "Foi a primeira vez que a Emília ficou sem resposta. A primeira e a última!", 1.0, 1.0],
        ["narrador", "Mais tarde, quando as duas dormiam, alguém bateu na porta.", 0.9, 1.0, "porta"],
        ["narrador", "A porta abriu. Quem empurrava com o focinho era o Rabicó.", 1.0, 0.4],
        ["rabico", "Narizinho, tem uma senhora aqui que quer entrar.", 1.0, 0.5],
        ["narrador", "Entrou uma formiga de saiote vermelho e avental de renda. Na cabeça, ela equilibrava uma bandeja de prata.", 1.0, 0.5],
        ["formiga", "Boa noite. Trago um presente da Rainha das Formigas para a Senhora Condessa de Três Estrelinhas.", 1.0, 0.6],
        ["narrador", "Narizinho sentiu ciúme, porque o presente não era para ela. Era para a boneca! Ela acordou a Emília com uma cotovelada.", 1.0, 0.5],
        ["narrador", "Na bandeja havia croquetes. São bolinhos compridos e fritos, sequinhos por fora, de dar água na boca.", 1.0, 0.5],
        ["emilia", "Diga à Rainha que muito obrigada. E que os croquetes estão lindos!", 1.05, 0.5],
        ["narizinho", "Presente se paga com presente, condessa. Mande para a rainha uma perninha daquele pernilongo que eu peguei hoje.", 1.0, 0.5],
        ["narrador", "A Emília enfeitou a perninha do mosquito com um laço de fita e pôs na bandeja. A formiga agradeceu e foi embora, contente.", 1.0, 0.8],
        ["narizinho", "Eu não como isso, Emília. Aposto que é croquete de minhoca!", 1.0, 0.5],
        ["emilia", "E daí? Você comeu linguiça no jantar e nem reclamou!", 1.05, 0.6],
        ["narrador", "As duas discutiram, discutiram. E ninguém prestou atenção no Rabicó, que foi chegando de mansinho...", 0.95, 0.8],
        ["narrador", "Quando a Emília foi pegar um croquete, a bandeja estava vazia!", 1.05, 0.5],
        ["emilia", "Cadê os meus croquetes? Quero os meus croquetes!", 1.1, 0.5],
        ["narrador", "O Rabicó tinha comido tudo, de uma vez só! A Emília pegou a vassoura e correu atrás do leitão pelo quarto.", 1.1, 0.6],
        ["narrador", "E Narizinho rolava na cama de tanto rir.", 1.0, 1.0],
        ["narrador", "Resumindo: a rainha mandou croquetes para a Emília, mas o guloso do Rabicó comeu todos.", 0.95, 0.8],
        ["narrador", "Enquanto ria, Narizinho teve uma ideia maluca: casar a Emília com o Rabicó!", 1.0, 0.8],
        ["narrador", "Mas antes disso, chegou o grande dia. Um cavalinho apareceu na porteira, trazendo alguém muito esperado.", 0.95, 1.0, "galope"],
        ["narrador", "Quem será? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    7: {"titulo": "Pedrinho chegou!", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo sete: Pedrinho chegou!", 0.95, 1.2],
        ["narrador", "No capítulo passado, a Rainha das Formigas mandou croquetes para a Emília. Mas o Rabicó comeu todos.", 1.0, 0.4],
        ["narrador", "E Narizinho teve uma ideia: casar a Emília com o Rabicó.", 1.0, 1.0],
        ["narrador", "Um dia antes, chegou uma carta de Pedrinho para Dona Benta. Ele dizia como queria ser recebido.", 1.0, 0.5],
        ["narrador", "Narizinho devia esperar na porteira, com a Emília de vestido novo e o Rabicó com um laço de fita no rabo.", 1.0, 0.5],
        ["narrador", "E Tia Nastácia devia fazer café com aqueles bolinhos de frigideira que só ela sabe fazer.", 1.0, 0.6],
        ["narrador", "Por isso, Narizinho acordou bem cedo. Vestiu a Emília de cor-de-rosa com bolinhas e amarrou duas fitas no Rabicó.", 1.0, 0.8],
        ["narrador", "Então, na estrada, apareceu um cavalinho trotando.", 1.0, 1.0, "galope"],
        ["narrador", "Em cima dele vinha Pedrinho, queimado de sol e alegre como um passarinho.", 1.05, 0.5],
        ["narizinho", "Viva! Desce depressa, Pedrinho, que temos mil coisas para conversar!", 1.1, 0.5],
        ["narrador", "Pedrinho desceu, abraçou a prima e escondeu um pacote atrás das costas.", 1.0, 0.4],
        ["pedrinho", "Adivinha o que eu trouxe para você!", 1.05, 0.5],
        ["narizinho", "Já sei. Uma boneca que chora e que abre e fecha os olhos.", 1.0, 0.6],
        ["narrador", "Pedrinho ficou de boca aberta, porque era isso mesmo!", 1.0, 0.4],
        ["pedrinho", "Como você adivinhou?", 1.0, 0.5],
        ["narizinho", "Porque eu conheço você, bobo. As meninas são muito mais espertas que os meninos.", 1.0, 0.5],
        ["pedrinho", "Mas não têm tanto muque!", 1.05, 0.4],
        ["narrador", "Muque é a força dos braços. Pedrinho dobrou o braço para mostrar o músculo duro, que ele treinava na escola.", 1.0, 0.5],
        ["pedrinho", "Com o meu muque e a sua esperteza, Narizinho, ninguém pode com a gente!", 1.05, 0.8],
        ["narrador", "Depois, ele deu os outros presentes. O Rabicó ganhou uma fita de seda e os restos do lanche. Gostou mais do lanche, claro.", 1.0, 0.5],
        ["narrador", "A Emília ganhou uma cozinha inteirinha de brinquedo: fogãozinho de lata, panelinhas e até um rolo de abrir massa.", 1.0, 0.8],
        ["narrador", "Até agora: Pedrinho chegou de cavalinho e trouxe presentes para todos.", 0.95, 0.8],
        ["narrador", "Dona Benta apareceu na varanda e Pedrinho correu para abraçar a avó.", 1.0, 0.5],
        ["narrador", "Logo estavam todos na mesa, ouvindo as novidades da cidade. Tia Nastácia trouxe a massa dos bolinhos para a sala, para não perder nenhuma palavra.", 1.0, 0.6],
        ["dona_benta", "Agora vão brincar, vocês dois. A Narizinho tem um mundo de coisas para contar.", 1.0, 0.8, "transicao"],
        ["narrador", "Os primos foram para o pomar, o lugar dos segredos e dos planos de aventura.", 1.0, 0.5],
        ["pedrinho", "Quando você me escreveu que a Emília falava, eu não acreditei. Mas ela fala mesmo!", 1.05, 0.5],
        ["narizinho", "E já é condessa! Mas eu quero mais. Quero fazer o Rabicó virar marquês e casar a Emília com ele.", 1.0, 0.6],
        ["narrador", "Marquês é um título importante, como conde e condessa. Pedrinho achou a ideia ótima.", 1.0, 0.8],
        ["narrador", "Depois, Narizinho contou tudo sobre o Reino das Águas Claras: o Príncipe Escamado, o palácio de conchas, o Doutor Caramujo.", 1.0, 0.5],
        ["narrador", "Pedrinho ficou com muita vontade de conhecer o príncipe. E não estranhou nada, porque para os dois primos as maravilhas eram uma coisa natural.", 1.0, 0.6],
        ["pedrinho", "Eu tenho uma ideia. E se os personagens das histórias saíssem dos livros e viessem brincar aqui no sítio?", 1.05, 0.6],
        ["narizinho", "Que lindo! Eu queria brincar com a Chapeuzinho Vermelho e com a Branca de Neve!", 1.05, 0.5],
        ["pedrinho", "E eu queria encontrar o Aladim, aquele da lâmpada mágica!", 1.05, 0.6],
        ["narrador", "Por que será que Pedrinho queria os personagens aqui no sítio?", 0.95, 2.5],
        ["narrador", "Porque ele queria viver aventuras novas, junto com eles, em vez de só ouvir as histórias de sempre.", 1.0, 0.6],
        ["nastacia", "Narizinho! Pedrinho! O café está na mesa!", 1.0, 0.5],
        ["narrador", "E os dois saíram correndo para comer os bolinhos de Tia Nastácia.", 1.05, 1.0],
        ["narrador", "Resumindo: Pedrinho chegou, adorou a Emília falante e quer trazer os personagens das histórias para o sítio.", 0.95, 0.8],
        ["narrador", "Naquela noite, todos dormiram tarde. E bem cedinho, antes do sol nascer, alguém bateu no vidro da janela de Narizinho...", 0.9, 0.8],
        ["narrador", "Era um convite para uma viagem a um reino muito diferente.", 0.9, 0.8],
        ["narrador", "Que reino será esse? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    8: {"titulo": "O ouro da Emília", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo oito: O ouro da Emília.", 0.95, 1.2],
        ["narrador", "No capítulo passado, Pedrinho chegou ao sítio, com presentes para todos.", 1.0, 0.4],
        ["narrador", "E bem cedinho, alguém bateu na janela de Narizinho, com um convite de viagem.", 1.0, 1.0],
        ["narrador", "Narizinho abriu a janela. Lá fora estava o Rabicó, montado num cavalinho de pau.", 1.0, 0.5],
        ["rabico", "Pula da cama, Narizinho! O sol já vai nascer. Está na hora de partir.", 1.05, 0.5],
        ["narrador", "Agora o leitão tinha um título: era o Marquês de Rabicó. E a Emília já esperava lá embaixo, num cavalinho malhado.", 1.0, 1.0, "magia"],
        ["narrador", "Para onde eles iam? Para um reino de bichinhos voadores. No dia anterior, um marimbondo tinha trazido um convite.", 1.0, 0.5],
        ["narrador", "Mas o papel estava rasgado bem no nome. Era da Rainha das Abelhas ou da Rainha das Vespas? Ninguém sabia.", 1.0, 0.5],
        ["narrador", "Para responder, a Emília inventou o borboletograma. Ela escreveu a resposta na asa de uma borboleta azul e mandou a borboleta voar.", 1.0, 0.5],
        ["narizinho", "Por que você não pôs o nome do Pedrinho, Emília?", 1.0, 0.4],
        ["emilia", "Porque ele não é nobre. Não é nem barão!", 1.05, 0.8],
        ["narrador", "Narizinho vestiu o vestido vermelho de bolso, para levar os bolinhos de Tia Nastácia. Montou no cavalinho dela, e lá foram todos pela estrada.", 1.0, 1.0, "galope"],
        ["narizinho", "Vamos apostar corrida, Emília? Pode ir na frente!", 1.05, 0.5],
        ["narrador", "A Emília saiu galopando. Mas Narizinho ficou para trás de propósito. Por que será?", 0.95, 2.5],
        ["narrador", "Porque ela queria conversar sozinha com o Rabicó sobre o casamento.", 1.0, 0.5],
        ["narizinho", "Então, marquês, quer ou não quer casar com a condessa?", 1.0, 0.4],
        ["rabico", "Caso, sim. Se me derem dois carrinhos cheios de milho, eu caso com quem vocês quiserem!", 1.05, 0.6],
        ["narrador", "Que guloso! Quando alcançaram a Emília, ela estava furiosa, porque tinha caído na brincadeira da corrida.", 1.0, 0.8],
        ["narrador", "Até agora: os três viajam para o reino das abelhas, ou das vespas, e o Rabicó só pensa em comida.", 0.95, 0.8],
        ["narrador", "De repente, o mato da beira do caminho se mexeu. Os cavalinhos empinaram, assustados.", 0.95, 0.5],
        ["narrador", "Saiu do mato um bando de lagartos. Na frente vinha o chefe: o Tom Mix, um caubói de cinema, com chapéu de abas largas.", 1.0, 0.5],
        ["narrador", "E sabe o que o Marquês de Rabicó fez? Pulou do cavalo e fugiu correndo, sem ajudar ninguém!", 1.05, 0.6],
        ["tom_mix", "Passem para cá o dinheiro!", 1.0, 0.6],
        ["narrador", "Narizinho ficou com medo, mas pensou depressa.", 1.0, 0.4],
        ["narizinho", "Dinheiro nós não temos, senhor Tom Mix. Mas tenho uns bolinhos muito gostosos. Quer um?", 1.0, 0.5],
        ["tom_mix", "Bolinho de ontem? Não quero. Eu quero ouro!", 1.0, 0.6],
        ["narrador", "Ouro é um metal amarelo e brilhante, que vale muito. Narizinho teve uma ideia genial.", 1.0, 0.5],
        ["narizinho", "Está bem. Vou dar ao senhor um montinho de ouro bem amarelo. Me empresta a sua bolsa e uma tesourinha.", 1.0, 0.6],
        ["narrador", "Ela cochichou alguma coisa no ouvido da Emília.", 0.95, 0.5],
        ["emilia", "Nunca! Prefiro morrer!", 1.1, 0.6],
        ["narrador", "Mas Narizinho insistiu tanto que a Emília, chorando, esticou a perna no colo da amiga.", 0.95, 0.5],
        ["narrador", "Narizinho descosturou a perna da boneca e despejou toda a macela na bolsa do bandido. A macela é amarelinha, parece pó de ouro!", 1.0, 0.6],
        ["tom_mix", "Muito bem! De hoje em diante, sou seu amigo, princesa. Se estiver em perigo, é só chamar: Mix, Mix, Mix! E eu apareço.", 1.0, 0.6],
        ["narrador", "E o caubói foi embora com os lagartos. Narizinho soltou o ar, aliviada.", 1.0, 0.6],
        ["emilia", "Ai! Agora não consigo andar nem montar com esta perna vazia!", 1.05, 0.5],
        ["narizinho", "Você me salvou, Emília. Obrigada! Vem na garupa do meu cavalinho, bem agarradinha.", 1.0, 0.8],
        ["narrador", "Resumindo: o Rabicó fugiu, e a Emília salvou todo mundo dando o recheio da própria perna como se fosse ouro.", 0.95, 0.8],
        ["narrador", "Então Narizinho lembrou do Rabicó, e ficou brava por ele ter abandonado as amigas no perigo. Ela chamou: Mix, Mix, Mix!", 1.0, 0.5],
        ["narizinho", "Tom Mix, o Rabicó fugiu e deixou a gente sozinha. Pegue esse medroso e dê um bom susto nele, para ele aprender!", 1.0, 0.8],
        ["narrador", "Será que o Tom Mix vai encontrar o Rabicó? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    9: {"titulo": "No reino das abelhas", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo nove: No reino das abelhas.", 0.95, 1.2],
        ["narrador", "No capítulo passado, o bandido Tom Mix parou Narizinho e a Emília na estrada. O Rabicó fugiu, com medo.", 1.0, 0.4],
        ["narrador", "A Emília deu a macela da perna como se fosse ouro. E Narizinho pediu ao Tom Mix um bom susto no Rabicó.", 1.0, 1.0],
        ["narrador", "O Tom Mix seguiu as pegadas do leitão até a floresta. E lá estava o Rabicó, com a cabeça enfiada dentro de uma abóbora enorme, comendo.", 1.0, 0.5],
        ["narrador", "O caubói chegou bem devagar, sem fazer barulho, e segurou o Rabicó pela perna.", 0.95, 0.5],
        ["rabico", "Coin, coin, coin! Me solta!", 1.1, 0.5],
        ["tom_mix", "Ordens da princesa do nariz arrebitado, senhor marquês. O senhor abandonou as amigas no perigo.", 1.0, 0.5],
        ["rabico", "Me dá cinco minutos? Preciso escrever uma carta.", 1.0, 0.5],
        ["narrador", "Uma libélula ia passando. Libélula é um inseto comprido, de quatro asas transparentes, que voa muito depressa.", 1.0, 0.5],
        ["rabico", "Amiga libélula, leve esta carta para a princesa Narizinho. Em troca, eu te dou um lindo lago azul!", 1.05, 0.5],
        ["narrador", "E a libélula saiu voando, rápida como um pensamento.", 1.05, 1.0, "transicao"],
        ["narrador", "Enquanto isso, Narizinho e a Emília chegavam ao palácio das abelhas. Era uma colmeia gigante, feita de cera, cheia de salinhas de seis lados.", 1.0, 1.0, "zumbido"],
        ["narrador", "Os zangões, que são as abelhas machos, vieram receber as duas. Eles contaram que o convite era mesmo da Rainha das Abelhas.", 1.0, 0.5],
        ["narrador", "E a Rainha das Vespas estava furiosa com Narizinho, por causa daquela vespa da jabuticaba. Ainda bem que elas não erraram o caminho!", 1.0, 0.6],
        ["narrador", "No palácio estava também um visitante conhecido: o Doutor Caramujo. Ele examinou a perna vazia da Emília.", 1.0, 0.5],
        ["caramujo", "Caso grave! Esta perna precisa de um tratamento muito científico.", 0.95, 0.5],
        ["narizinho", "Que nada, doutor. Tia Nastácia põe macela nova, costura com agulha e linha, e pronto.", 1.0, 0.5],
        ["caramujo", "Tia Nastácia? Uma curandeira qualquer! Isso não é ciência!", 0.95, 0.5],
        ["narizinho", "Não fale assim da Tia Nastácia! Ela cuida de todo mundo lá no sítio, e cuida muito bem. Peça desculpas!", 1.05, 0.6],
        ["narrador", "O Doutor Caramujo ficou envergonhado e foi embora de fininho. E os zangões arranjaram duas muletinhas para a Emília andar.", 1.0, 0.8],
        ["narrador", "Depois, Narizinho encontrou o Major Agarra, que tinha vindo do Reino das Águas Claras.", 1.0, 0.4],
        ["major", "O príncipe anda muito triste, menina. Não quer casar com nenhuma princesa. Dizem que ele gosta de alguém... mas ninguém sabe quem é.", 0.95, 0.6],
        ["narrador", "O coração de Narizinho bateu mais depressa. Por que será?", 0.95, 2.5],
        ["narrador", "Porque ela desconfiou que esse alguém podia ser ela mesma!", 1.0, 0.5],
        ["narizinho", "Major, diga ao príncipe que eu estou bem. E que um dia vou convidar ele para visitar o sítio.", 1.0, 0.8],
        ["narrador", "Até agora: o Tom Mix pegou o Rabicó, a Emília ganhou muletas, e o príncipe está triste de saudade.", 0.95, 0.8],
        ["narrador", "Nessa hora chegou a libélula com a carta do Rabicó. A letra era torta e as palavras estavam todas erradas.", 1.0, 0.5],
        ["narrador", "A carta dizia: Desculpe a minha covardia. O Tom Mix me pegou. Tenha dó deste infeliz.", 0.95, 0.6],
        ["narizinho", "Esse Rabicó! Está bem, eu perdoo. Leve este bilhete para o Tom Mix, depressa!", 1.0, 0.5],
        ["narrador", "O Tom Mix leu o perdão e soltou o Rabicó. Mas a libélula cobrou o lago azul.", 1.0, 0.5],
        ["rabico", "Lago? Você pode se afogar num lago. Tome esta sementinha de abóbora, que é melhor!", 1.05, 0.5],
        ["narrador", "A libélula ficou furiosa, porque promessa é promessa.", 1.0, 0.8],
        ["narrador", "No palácio, uma abelhinha explicou como era a vida da colmeia.", 1.0, 0.4],
        ["abelha", "Aqui ninguém precisa mandar. Cada uma já nasce sabendo o seu trabalho. Todas buscam o mel das flores e cuidam da casa.", 1.0, 0.5],
        ["abelha", "E a rainha não é uma rainha emproada. Ela é a mãe de todas nós.", 1.0, 0.6],
        ["narrador", "Então Narizinho conheceu a rainha, sentada num trono de cera.", 1.0, 0.4],
        ["rainha", "Seja bem-vinda, menina. Leve todo o mel que quiser. Aqui o dinheiro não vale nada.", 0.95, 0.6],
        ["narrador", "Narizinho escolheu mel de flor de jabuticabeira, para a avó, e cera bem branquinha, para Tia Nastácia.", 1.0, 0.5],
        ["narrador", "O Tom Mix chegou com uma fila de grilos, cada um carregando dois barrilzinhos nas costas. E encheram os barris de mel.", 1.0, 1.0],
        ["narrador", "Resumindo: Narizinho perdoou o Rabicó, conheceu a rainha das abelhas e ganhou mel para levar para casa.", 0.95, 0.8],
        ["narrador", "Mas na volta, a Emília, que ia na frente dentro de um barril vazio, ficou de pé e gritou:", 0.95, 0.5],
        ["emilia", "Estou vendo um monstro lá na frente! Com cabeça de porco e pés de tartaruga!", 1.1, 0.8],
        ["narrador", "Que monstro será esse? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    10: {"titulo": "O feitiço da velha", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo dez: O feitiço da velha.", 0.95, 1.2],
        ["narrador", "No capítulo passado, Narizinho visitou o reino das abelhas e ganhou mel. O Tom Mix soltou o Rabicó.", 1.0, 0.4],
        ["narrador", "Na volta, a Emília viu um monstro com cabeça de porco e pés de tartaruga.", 1.0, 1.0],
        ["narrador", "O Tom Mix foi na frente para ver o que era. Logo voltou, rindo.", 1.0, 0.5],
        ["tom_mix", "Não é monstro nenhum, princesa! É o marquês, montado num jabuti!", 1.05, 0.5],
        ["narrador", "Jabuti é uma tartaruga que vive na terra. Anda muito devagar e tem um casco duro nas costas.", 1.0, 0.5],
        ["narrador", "E o Rabicó gritava com o coitado do jabuti, para ele andar mais depressa.", 1.0, 0.5],
        ["rabico", "Anda, jabuti preguiçoso! Mais depressa!", 1.05, 0.6],
        ["narrador", "Narizinho ficou indignada, porque o Rabicó estava maltratando um bicho mais fraco que ele.", 1.0, 0.4],
        ["narizinho", "Desce já daí, Rabicó! Agora é a sua vez de carregar. Senhor jabuti, pode montar no marquês!", 1.05, 0.6],
        ["narrador", "O jabuti, bem devagarinho, subiu nas costas do Rabicó. E o Rabicó teve que carregar o jabuti pelo caminho.", 1.0, 0.5],
        ["emilia", "Bem feito! Isso é por ter comido os meus croquetes!", 1.1, 0.5],
        ["narrador", "E uma vozinha fina passou voando lá no alto. Era a libélula.", 1.0, 0.4],
        ["libelula", "E isso é pelo meu lago azul!", 1.1, 0.8],
        ["narrador", "Nesse dia, o Rabicó perdeu pelo menos um quilo e aprendeu um pouco a respeitar os outros.", 1.0, 1.0, "transicao"],
        ["narrador", "Perto do sítio, todos pararam para almoçar debaixo de uma figueira. O almoço era só mel.", 1.0, 0.5],
        ["narrador", "Então um passarinho cantou no galho. Narizinho olhou bem para ele.", 1.0, 0.4],
        ["narizinho", "Emília, esse passarinho não tem a cara do Pedrinho?", 1.0, 0.5],
        ["narrador", "O passarinho desceu e pousou no ombro dela.", 1.0, 0.4],
        ["pedrinho", "Sou eu mesmo, Narizinho! Apareceu lá no sítio uma velha de bengala e cesta no braço. Ela perguntou pela menina do nariz arrebitado.", 1.0, 0.5],
        ["pedrinho", "Eu respondi mal para ela. Então ela me virou passarinho! E virou a vovó numa tartaruga, e a Tia Nastácia numa galinha!", 1.05, 0.6],
        ["narizinho", "Que horror! O que vai ser de nós agora?", 1.05, 0.6],
        ["tom_mix", "Deixe comigo, princesa. Eu vou encontrar essa velha e trazer ela de volta, para desfazer o feitiço.", 1.0, 0.8],
        ["narrador", "Até agora: o Rabicó levou uma lição, mas uma velha transformou a família de Narizinho em bichos.", 0.95, 0.8],
        ["narrador", "Narizinho voltou para o sítio, muito triste, com o passarinho Pedrinho no ombro.", 0.95, 0.5],
        ["narrador", "Na cozinha, uma galinha cacarejava. Na sala, uma tartaruga de óculos costurava na rede.", 0.95, 0.5],
        ["narizinho", "Vovó! Não me conhece mais, vovó?", 0.95, 0.8],
        ["narrador", "A tartaruga ficou quieta, quieta. Narizinho chorou, abraçada com a Emília.", 0.9, 0.8],
        ["narrador", "Passou um dia. Passaram dois dias. E nada do Tom Mix.", 0.9, 0.6],
        ["emilia", "Calma, Narizinho. A conta para tudo é três. No terceiro dia ele aparece.", 1.0, 0.8],
        ["narrador", "No terceiro dia, as duas ficaram na janela, olhando a estrada. Por que será que a Emília tinha tanta esperança?", 0.95, 2.5],
        ["narrador", "Porque o Tom Mix tinha prometido ajudar. E a Emília achava que ele era como nos filmes, onde o herói sempre chega no fim.", 1.0, 0.6],
        ["narrador", "De repente, um cachorrinho latiu no terreiro. Era o Tom Mix! E na frente dele vinha a velha, de braços para cima.", 1.05, 0.5],
        ["tom_mix", "Agora você vai desfazer o feitiço que fez!", 1.0, 0.6],
        ["narrador", "Narizinho fechou os olhos, com medo da cara da velha. Depois foi abrindo, bem devagarinho...", 0.85, 1.2, "magia"],
        ["narrador", "E quem ela viu? A Tia Nastácia, de verdade, sorrindo para ela!", 1.0, 0.4],
        ["nastacia", "Acorda, menina! Parece que você estava sonhando.", 0.95, 0.6],
        ["narrador", "Narizinho sentou na cama, esfregando os olhos. Tudo tinha sido um sonho!", 1.0, 0.5],
        ["narizinho", "E a vovó? E o Pedrinho? E o Tom Mix?", 1.05, 0.5],
        ["nastacia", "A sua avó está lá dentro, costurando. O Pedrinho está brincando no quintal. E deixe de bobagem, que o café está esfriando!", 1.0, 1.0],
        ["narrador", "Resumindo: a viagem ao reino das abelhas, o bandido e o feitiço foram um sonho comprido de Narizinho.", 0.95, 0.8],
        ["narrador", "Mas uma coisa não era sonho: Narizinho ainda queria casar a Emília com o Rabicó. E a Emília não queria de jeito nenhum!", 1.0, 0.6],
        ["narrador", "Como será que Narizinho vai convencer a boneca? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    11: {"titulo": "O Visconde de Sabugosa", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo onze: O Visconde de Sabugosa.", 0.95, 1.2],
        ["narrador", "No capítulo passado, Narizinho sonhou uma aventura comprida: o reino das abelhas, o bandido Tom Mix e o feitiço da velha.", 1.0, 0.4],
        ["narrador", "Quando acordou, ainda tinha um plano: casar a Emília com o Rabicó.", 1.0, 1.0],
        ["narrador", "O Rabicó era um leitãozinho ruivo, com manchas brancas e um rabinho bem curto. Por isso o nome dele era Rabicó.", 1.0, 0.5],
        ["narrador", "No sítio, quando os leitões ficavam gordinhos, Tia Nastácia levava para a cozinha, para virar comida. Mas o Rabicó tinha uma amiga que protegia ele: Narizinho.", 1.0, 0.6],
        ["narrador", "Uma tarde, Narizinho ouviu Dona Benta falar com Tia Nastácia.", 1.0, 0.4],
        ["dona_benta", "Amanhã é aniversário do Pedrinho. Vamos fazer um jantar especial, com leitão assado.", 1.0, 0.5],
        ["narrador", "Narizinho entendeu tudo e correu atrás do Rabicó. Por que será que ela correu?", 0.95, 2.5],
        ["narrador", "Porque o único leitão gordinho no sítio era o Rabicó! E ela não queria que o amigo virasse jantar.", 1.0, 0.5],
        ["narizinho", "Vem comigo, Rabicó. Vou esconder você num lugar que só eu sei. E nada de barulho!", 1.0, 0.5],
        ["narrador", "Ela deixou o leitão escondido, com milho, uma abóbora e uma lata de água. Tia Nastácia procurou o sítio inteiro e não achou.", 1.0, 0.5],
        ["narrador", "No jantar do aniversário, comeram galinha. E no dia seguinte, o Rabicó voltou para o terreiro, todo tranquilo.", 1.0, 0.5],
        ["nastacia", "Olha só quem apareceu! Escapou desta vez, seu danadinho!", 1.0, 1.0, "transicao"],
        ["narrador", "Agora, Narizinho podia cuidar do casamento.", 1.0, 0.4],
        ["narizinho", "Senhora condessa, está na hora de casar. Que tal o Marquês de Rabicó?", 1.0, 0.5],
        ["emilia", "Nunca! Ele é um medroso. Fugiu quando o bandido apareceu!", 1.05, 0.6],
        ["narizinho", "Mas você não sabe o segredo. O Rabicó é um príncipe! Uma fada má virou ele em porco.", 1.0, 0.5],
        ["narizinho", "Ele só vai virar príncipe de novo quando achar um anel mágico, escondido na barriga de uma minhoca. Por isso ele vive cavando a terra!", 1.0, 0.6],
        ["narrador", "Era tudo invenção de Narizinho. Mas a Emília sonhava ser princesa. E pensou, pensou, pensou.", 0.95, 0.5],
        ["emilia", "Está bem, eu aceito. Mas só vou morar com ele quando ele virar príncipe!", 1.05, 0.8],
        ["narrador", "Narizinho correu até o pomar, onde estava Pedrinho.", 1.05, 0.4],
        ["narizinho", "Pedrinho, depressa! Faz um visconde de sabugo, com cartola e uma marca de coroa na testa. Ele vai ser o pai do Rabicó!", 1.05, 0.6],
        ["narrador", "Sabugo é o miolo da espiga de milho, depois que tiram os grãos. É comprido e duro, do tamanho de uma mão.", 1.0, 0.5],
        ["narrador", "Pedrinho pegou um sabugo e fez braços, pernas e cara. As palhinhas do milho viraram barba. E na cabeça, uma cartolinha, que é um chapéu alto e redondo.", 1.0, 0.8],
        ["narrador", "Até agora: Narizinho salvou o Rabicó, enganou a Emília, e Pedrinho fez um visconde de sabugo de milho.", 0.95, 0.8],
        ["narrador", "Pedrinho levou o visconde até o quarto da Emília e bateu na porta.", 1.0, 0.8, "porta"],
        ["visconde", "Sou o Visconde de Sabugosa. Venho pedir a mão da Condessa de Três Estrelinhas para o meu filho, o Marquês de Rabicó.", 0.95, 0.6],
        ["narizinho", "Entre, senhor visconde. E como vai a sua senhora mãe, Dona Palha-de-Milho?", 1.0, 0.5],
        ["visconde", "Coitada. Foi comida pela vaca.", 0.9, 0.6],
        ["narizinho", "Que pena. Mas a vida é assim: a vaca come o milho, e a gente come... bom, deixa para lá!", 1.0, 0.8],
        ["narrador", "A Emília entrou, de vestido de bolinhas. O visconde tirou a cartola para cumprimentar, e ela viu a marca de coroa na testa dele.", 1.0, 0.5],
        ["narizinho", "A Emília sabe fazer tudo, visconde. Até este vestido foi ela que costurou!", 1.0, 0.5],
        ["emilia", "Não fui eu, não. Foi a Tia Nastácia.", 1.0, 0.6],
        ["narrador", "A Emília ainda não sabia mentir. E isso é uma coisa boa!", 1.0, 0.5],
        ["visconde", "O meu filho tem bom gênio. Gosta de dormir no sol e de cavar a terra atrás de minhocas. O único defeito é comer tudo o que encontra.", 0.95, 0.5],
        ["emilia", "Comigo, ele só vai comer coisas limpinhas e cheirosas!", 1.05, 0.5],
        ["narizinho", "Então, Emília? Aceita ou não aceita o Marquês de Rabicó?", 1.0, 0.8],
        ["narrador", "A Emília pensou, pensou, e respondeu:", 0.95, 0.6],
        ["emilia", "Quero!", 1.05, 1.0, "festa"],
        ["narrador", "O visconde deu um beijo na testa da noiva. E a Emília, vermelhinha de vergonha, saiu correndo para o quarto.", 1.0, 1.0],
        ["narrador", "Resumindo: a Emília aceitou casar com o Rabicó, porque acha que ele é um príncipe encantado.", 0.95, 0.8],
        ["narrador", "Agora vem o noivado, com doces e festa. Mas o Rabicó é muito guloso. Será que ele vai se comportar?", 1.0, 0.6],
        ["narrador", "Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    12: {"titulo": "O casamento da Emília", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo doze: O casamento da Emília.", 0.95, 1.2],
        ["narrador", "No capítulo passado, Narizinho contou para a Emília que o Rabicó era um príncipe encantado.", 1.0, 0.4],
        ["narrador", "Pedrinho fez o Visconde de Sabugosa, de sabugo de milho. E a Emília aceitou o pedido de casamento.", 1.0, 1.0],
        ["narrador", "Começou o noivado. Noivado é o tempo antes do casamento, quando os noivos se visitam.", 1.0, 0.5],
        ["narrador", "Toda tarde, Pedrinho trazia o Rabicó para visitar a noiva. Mas o leitão só cheirava o chão, procurando comida. Nem ouvia a conversa!", 1.0, 0.5],
        ["pedrinho", "Assim não dá. O Rabicó não sabe brincar. Vou arranjar alguém para ficar no lugar dele.", 1.0, 0.5],
        ["narrador", "Pedrinho achou no quintal um vidro azul, vazio, e pôs uma cartolinha nele. Era o Senhor Vidro Azul, que ia namorar no lugar do noivo.", 1.0, 0.6],
        ["narrador", "E o Rabicó voltou para o pomar, para cavar a terra atrás da tal minhoca do anel.", 1.0, 0.8],
        ["pedrinho", "O marquês mandou construir um castelo de ouro para a noiva. Com um lago na frente!", 1.0, 0.5],
        ["emilia", "Um lago? Aposto que é uma lata de água!", 1.1, 0.6],
        ["narrador", "Pedrinho, fazendo a voz do Vidro Azul, leu uns versinhos que o marquês tinha mandado:", 1.0, 0.5],
        ["pedrinho", "Pirolito que bate, bate. Pirolito que já bateu. Quem adora o marquês é ela. Quem adora a Emília sou eu.", 0.95, 0.8],
        ["emilia", "Está errado! Eu vou casar, mas não adoro ninguém. Já pensou, adorar um leitão?", 1.05, 0.8],
        ["narrador", "Uma semana depois, Narizinho pediu a Dona Benta umas moedinhas para comprar os doces da festa. E Dona Benta deu.", 1.0, 0.8, "transicao"],
        ["narrador", "Até agora: o noivo comilão foi trocado por um vidro azul, e a Emília continua bem desconfiada.", 0.95, 0.8],
        ["narrador", "Chegou o dia do casamento. A mesa ficou debaixo de uma laranjeira, com seis cocadas, seis pés-de-moleque e uma rapadura.", 1.0, 0.5],
        ["narrador", "Os convidados eram de mentirinha: pedras, tijolos e pedaços de pau, cada um fazendo o papel de uma pessoa.", 1.0, 0.5],
        ["narrador", "A Emília veio de vestido branco e véu. O Rabicó veio de cartola, com uma faixa de seda no pescoço.", 1.0, 0.5],
        ["narrador", "Mas quando o Rabicó sentiu o cheiro das cocadas, ficou com água na boca e não viu mais nada.", 1.0, 0.5],
        ["narrador", "Os noivos se casaram. Narizinho pôs um anelzinho no braço da Emília, porque a boneca não tem dedos.", 1.0, 1.0, "festa"],
        ["narrador", "Pedrinho pôs no Rabicó uma aliança de casca de laranja. E o Rabicó tentou comer a aliança duas vezes!", 1.05, 0.6],
        ["narrador", "Depois, os primos se distraíram conversando. E ninguém vigiou a mesa dos doces...", 0.95, 0.8],
        ["narrador", "O que será que o Rabicó fez?", 0.95, 2.5],
        ["narrador", "Nhoc! Ele abocanhou a cocada mais bonita e saiu correndo pelo campo, com a cocada na boca!", 1.1, 0.5],
        ["narrador", "Pedrinho correu atrás dele, furioso. A festa virou uma bagunça.", 1.05, 0.5],
        ["emilia", "Eu sabia! Eu não queria casar com esse guloso! Ele não respeita a esposa!", 1.1, 0.5],
        ["narizinho", "Calma, Emília. Um dia ele vira príncipe, lembra?", 1.0, 0.5],
        ["pedrinho", "Que príncipe nada! A Narizinho enganou você. O Rabicó é porco e vai ser porco para sempre.", 1.05, 0.6],
        ["narrador", "Quando ouviu isso, a Emília caiu para trás, desmaiada.", 1.0, 1.0],
        ["narrador", "Desde esse dia, a Emília ficou casada com o Rabicó, mas cada um morando no seu canto. Ela nunca perdoou a mentira.", 1.0, 0.8, "transicao"],
        ["narrador", "Chegou o Ano-Bom, que é o primeiro dia do ano. Dona Benta fez um jantar grande, com muitos pratos.", 1.0, 0.5],
        ["narrador", "E de repente, na mesa, apareceu um leitão assado numa travessa.", 0.95, 0.6],
        ["narrador", "Narizinho procurou o Rabicó no terreiro inteiro e não achou. Voltou chorando.", 0.95, 0.4],
        ["narizinho", "Não come esse leitão, Pedrinho! É o Rabicó!", 1.05, 0.6],
        ["narrador", "Pedrinho ficou com os olhos cheios de lágrimas. Mas a Emília bateu palmas!", 1.0, 0.4],
        ["emilia", "Estou viúva! Agora posso casar com o visconde!", 1.1, 0.6],
        ["narrador", "Viúva é quem perdeu o marido. Narizinho ficou tão brava que deu um peteleco na boneca.", 1.0, 0.5],
        ["narizinho", "Vou pedir ao Doutor Caramujo para pôr nela o que está faltando!", 1.05, 0.4],
        ["dona_benta", "E o que está faltando na Emília, minha filha?", 1.0, 0.5],
        ["narizinho", "Coração, vovó! A Emília não tem nem um pedacinho de coração!", 1.05, 0.8],
        ["narrador", "Mas quantas lágrimas à toa! O leitão da travessa não era o Rabicó.", 1.0, 0.5],
        ["narrador", "O espertinho tinha fugido para o mato na véspera. E depois voltou para o terreiro, muito tranquilo, como se nada tivesse acontecido.", 1.0, 1.0],
        ["narrador", "Resumindo: o casamento da Emília deu errado, e o Rabicó escapou de novo do forno.", 0.95, 0.8],
        ["narrador", "Enquanto isso, lá no fundo do riacho, o Príncipe Escamado estava doente de tristeza. E ele ia mandar um pedido muito importante para Narizinho...", 0.9, 0.8],
        ["narrador", "Que pedido será? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
}

for n, c in CAPITULOS.items():
    palavras = sum(len(t[1].split()) for t in c["trechos"])
    print(f"Capítulo {n}, {c['titulo']}: {len(c['trechos'])} trechos, cerca de "
          f"{palavras / 130 + sum(t[3] for t in c['trechos']) / 60:.1f} min.")

In [ ]:
# ── 6. Criar as vozes e ouvir (troque a semente de quem não ficou bom) ──────
import hashlib, os, shutil, requests
from IPython.display import Audio, display

def gerar(texto, destino, **campos):
    arquivos = campos.pop("files", None)
    dados = {"text": texto, "language": "pt", "effect_preset": "raw", **campos}
    r = requests.post(f"{BASE}/generate", data=dados, files=arquivos, timeout=1800)
    if r.status_code != 200:
        raise SystemExit(f"Erro {r.status_code} em /generate: {r.text[:500]}\nLog: {LOG_PATH}")
    with open(destino, "wb") as f:
        f.write(r.content)

def chave(*partes):
    return hashlib.sha1(repr(partes).encode()).hexdigest()[:8]

REF = {}   # personagem -> (wav de referência, texto)
for nome, v in VOZES.items():
    # A fala mais longa do personagem no primeiro capítulo em que ele aparece.
    texto = next(max(falas, key=len) for _, c in sorted(CAPITULOS.items())
                 if (falas := [t[1] for t in c["trechos"] if t[0] == nome]))
    arq = f"{nome}_{chave(v, texto)}.wav"
    wav = os.path.join(PASTA_VOZES, arq)
    antigo = os.path.join(LIVRO, "cap01", "vozes", arq)   # versão do primeiro teste
    if not os.path.exists(wav) and os.path.exists(antigo):
        shutil.copy(antigo, wav)
    if not os.path.exists(wav):
        if v.get("amostra"):
            with open(v["amostra"], "rb") as a:
                gerar(texto, wav, seed=v["semente"], files={"ref_audio": ("amostra.wav", a)})
        else:
            gerar(texto, wav, instruct=v["voz"], seed=v["semente"])
    REF[nome] = (wav, texto)
    print(f"{nome} ({v['voz']}, semente {v['semente']}): {texto}")
    display(Audio(wav))

In [ ]:
# ── 7. Narrar e montar cada capítulo (A: só voz; B: vinheta, sons leves, riacho) 
import os, subprocess
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

def narrar(n):
    pasta = os.path.join(LIVRO, f"cap{n:02d}", "trechos")
    os.makedirs(pasta, exist_ok=True)
    arquivos = []
    for i, t in enumerate(CAPITULOS[n]["trechos"], 1):
        quem, texto, vel, pausa = t[:4]
        efeito = t[4] if len(t) > 4 else None
        ref_wav, ref_texto = REF[quem]
        wav = os.path.join(pasta, f"{i:03d}_{quem}_{chave(os.path.basename(ref_wav), texto, vel)}.wav")
        if not os.path.exists(wav):
            with open(ref_wav, "rb") as ref:
                gerar(texto, wav, seed=VOZES[quem]["semente"], ref_text=ref_texto, speed=vel,
                      files={"ref_audio": ("ref.wav", ref, "audio/wav")})
            print(f"{n}.{i:03d} {quem}: {texto}")
        arquivos.append((wav, pausa, efeito))
    return arquivos

def tom(freq, dur, vol, sr):
    t = np.arange(int(dur * sr)) / sr
    return (vol * np.sin(2 * np.pi * freq * t) * np.exp(-3 * t)).astype("float32")

def vinheta(sr):
    # Três notas suaves (dó, mi, sol), sem ataque forte.
    v = np.zeros(int(2.2 * sr), dtype="float32")
    for k, f in enumerate([523.25, 659.25, 783.99]):
        nota = tom(f, 1.4, 0.12, sr)
        ini = int(k * 0.35 * sr)
        v[ini:ini + len(nota)] += nota
    return v

def riacho(n, sr):
    # Ruído suave filtrado, que lembra água correndo, bem baixo e com entrada lenta.
    rng = np.random.default_rng(7)
    r = np.convolve(rng.standard_normal(n + 400).astype("float32"),
                    np.ones(40, dtype="float32") / 40, mode="valid")[:n]
    r *= 0.01 * (1 + 0.3 * np.sin(2 * np.pi * 0.15 * np.arange(n) / sr))
    fade = min(n // 2, 4 * sr)
    r[:fade] *= np.linspace(0, 1, fade)
    r[-fade:] *= np.linspace(1, 0, fade)
    return r

# ── Sonoplastia leve (só na versão B) ────────────────────────────────────────
# Sete sons curtos, sintetizados aqui mesmo, sempre iguais (mesmo som = mesmo sentido).
# Entram só nas pausas, nunca por cima da fala; ataque suave; pico baixo.

def _env(n, sr, ataque=0.03, queda=4.0):
    t = np.arange(n) / sr
    a = np.clip(t / ataque, 0, 1)
    return (a * np.exp(-queda * t)).astype("float32")

def _nota(f, dur, vol, sr, queda=3.0):
    t = np.arange(int(dur * sr)) / sr
    som = np.sin(2 * np.pi * f * t) + 0.25 * np.sin(2 * np.pi * 2 * f * t)
    return (vol * som * _env(len(t), sr, 0.02, queda)).astype("float32")

def _suave(x, k):
    return np.convolve(x, np.ones(k, dtype="float32") / k, mode="same").astype("float32")

def _junta(sr, dur, pecas):
    out = np.zeros(int(dur * sr), dtype="float32")
    for ini, p in pecas:
        i = int(ini * sr)
        p = p[:len(out) - i]
        out[i:i + len(p)] += p
    return out

def ef_transicao(sr):   # troca de cena: duas notas que descem (sol, dó)
    return _junta(sr, 1.3, [(0, _nota(783.99, 1.0, 0.07, sr)), (0.3, _nota(523.25, 1.0, 0.07, sr))])

def ef_magia(sr):       # encanto, sonho, feitiço: quatro notas agudas subindo
    fs = [1046.5, 1318.5, 1568.0, 2093.0]
    return _junta(sr, 1.5, [(k * 0.12, _nota(f, 0.9, 0.035, sr, 4.0)) for k, f in enumerate(fs)])

def ef_agua(sr):        # mergulho / água: ruído abafado + duas bolhas
    rng = np.random.default_rng(11)
    n = int(0.6 * sr)
    chua = _suave(rng.standard_normal(n).astype("float32"), 60) * _env(n, sr, 0.04, 6.0) * 0.6
    def bolha(f0):
        t = np.arange(int(0.12 * sr)) / sr
        return (0.04 * np.sin(2 * np.pi * (f0 + 900 * t) * t) * _env(len(t), sr, 0.01, 20)).astype("float32")
    return _junta(sr, 0.9, [(0, chua), (0.35, bolha(500)), (0.55, bolha(650))])

def ef_galope(sr):      # cavalinho: pac, pac, pac (quatro batidinhas de madeira)
    def pac(f):
        t = np.arange(int(0.08 * sr)) / sr
        return (0.09 * np.sin(2 * np.pi * f * t) * _env(len(t), sr, 0.003, 45)).astype("float32")
    return _junta(sr, 1.2, [(k * 0.25, pac(700 if k % 2 else 600)) for k in range(4)])

def ef_porta(sr):       # batida na porta: toc, toc, toc (grave e abafado)
    def toc():
        t = np.arange(int(0.12 * sr)) / sr
        return (0.12 * np.sin(2 * np.pi * 160 * t) * _env(len(t), sr, 0.004, 30)).astype("float32")
    return _junta(sr, 1.0, [(k * 0.28, toc()) for k in range(3)])

def ef_zumbido(sr):     # abelhas: zum-zum baixinho, entra e sai devagar
    t = np.arange(int(1.2 * sr)) / sr
    z = sum(np.sin(2 * np.pi * h * 210 * t) / h for h in (1, 2, 3))
    z *= 0.5 + 0.5 * np.sin(2 * np.pi * 9 * t)
    janela = np.sin(np.pi * t / t[-1]) ** 2
    return (0.025 * z * janela).astype("float32")

def ef_festa(sr):       # alegria, festa, final feliz: acorde curto (dó, mi, sol juntos)
    return _junta(sr, 1.4, [(0, _nota(f, 1.3, 0.04, sr, 2.5)) for f in (523.25, 659.25, 783.99)])

EFEITOS = {"transicao": ef_transicao, "magia": ef_magia, "agua": ef_agua, "galope": ef_galope,
           "porta": ef_porta, "zumbido": ef_zumbido, "festa": ef_festa}

def exportar(audio, sr, pasta, nome):
    tmp = os.path.join(pasta, nome + ".wav")
    sf.write(tmp, audio, sr)
    mp3 = os.path.join(pasta, nome + ".mp3")
    # Volume normalizado para fone e caixinha: -16 LUFS, pico máximo -1.5 dBTP.
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", tmp,
                    "-af", "loudnorm=I=-16:TP=-1.5:LRA=11", "-ar", "44100",
                    "-codec:a", "libmp3lame", "-q:a", "2", mp3], check=True)
    print(f"{nome}.mp3: {len(audio) / sr / 60:.1f} min")
    display(Audio(mp3))

def montar(n, arquivos):
    partes, marcas, sr, pos = [], [], None, 0
    for wav, pausa, efeito in arquivos:
        a, s = sf.read(wav, dtype="float32", always_2d=True)
        sr = sr or s
        assert s == sr, f"taxa de amostragem diferente em {wav}"
        fala = a.mean(axis=1)
        pos += len(fala)
        if efeito:
            som = EFEITOS[efeito](sr)
            pausa = max(pausa, len(som) / sr + 0.4)   # o som cabe inteiro na pausa
            marcas.append((pos + int(0.15 * sr), som))
        partes += [fala, np.zeros(int(pausa * sr), dtype="float32")]
        pos += int(pausa * sr)
    voz = np.concatenate(partes)
    # Sons das pausas: 15 dB abaixo da fala (RMS) e pico até 1/3 do pico da fala.
    ativo = voz[np.abs(voz) > 0.01]
    rms_voz = float(np.sqrt((ativo ** 2).mean())) if len(ativo) else 0.1
    sons = np.zeros_like(voz)
    for ini, som in marcas:
        sa = som[np.abs(som) > 1e-4]
        g = rms_voz * 10 ** (-15 / 20) / float(np.sqrt((sa ** 2).mean()))
        g = min(g, np.abs(voz).max() / 3 / float(np.abs(som).max()))
        sons[ini:ini + len(som)] += (som * g)[:len(sons) - ini]
    fundo = riacho(len(voz), sr) if CAPITULOS[n].get("ambiente", "riacho") == "riacho" else 0
    um_seg = np.zeros(sr, dtype="float32")
    pasta = os.path.join(LIVRO, f"cap{n:02d}")
    exportar(np.concatenate([um_seg, voz, um_seg]), sr, pasta, f"cap{n:02d}_A_so_voz")
    exportar(np.concatenate([vinheta(sr), um_seg, voz + sons + fundo, um_seg, vinheta(sr)]),
             sr, pasta, f"cap{n:02d}_B_com_ambiente")

REFAZER = False   # True: narra de novo capítulos que já têm os dois MP3 prontos

for n in CAPITULOS_PARA_GERAR:
    print(f"\n=== Capítulo {n}: {CAPITULOS[n]['titulo']} ===")
    prontos = [os.path.join(LIVRO, f"cap{n:02d}", f"cap{n:02d}_{v}.mp3")
               for v in ("A_so_voz", "B_com_ambiente")]
    if not REFAZER and all(map(os.path.exists, prontos)):
        print("Já estava pronto (para refazer, use REFAZER = True).")
        continue
    montar(n, narrar(n))
print("\nPronto! Os arquivos estão no Drive, em", LIVRO)

# Se um trecho ficou ruim, apague o .wav dele na pasta capNN/trechos/ e rode esta célula de novo.